# NLP701@MBZUAI Fall 2025 - Lab 03


## Today's Agenda
- Implement a POS tagger
  - Working with Universal Dependencies data
  - Implementing HMM model for POS tagging

# **Data Preprocessing**



## **Universal Dependencies**
![image](https://avatars.githubusercontent.com/u/7457237?s=200&v=4)

[Universal Dependencies (UD)](https://universaldependencies.org/) is a framework for consistent annotation of grammar (parts of speech, morphological features, and syntactic dependencies) across different human languages.
UD is an open community effort with over 300 contributors producing nearly 200 treebanks in over 100 languages.

Paper: [Universal Dependencies v1: A Multilingual Treebank Collection](https://aclanthology.org/L16-1262/) (Nivre et al., LREC 2016)

In [1]:
# Download UD_English-GUM from the GitHub repository
!git clone https://github.com/UniversalDependencies/UD_English-GUM.git

Cloning into 'UD_English-GUM'...
remote: Enumerating objects: 8066, done.
remote: Counting objects: 100% (692/692), done.
remote: Compressing objects: 100% (312/312), done.
remote: Total 8066 (delta 543), reused 430 (delta 380), pack-reused 7374 (from 1)
Receiving objects: 100% (8066/8066), 77.34 MiB | 10.01 MiB/s, done.
Resolving deltas: 100% (7439/7439), done.


## Looking at the data using Unix commands

It is not always efficient to write a Python script **just to get basic corpus statistics**, such as the number of sentences, tokens, labels etc.
In this exercise, we will learn how to get these statistics quickly using **Unix commands**.
This is one of the basic skills that you're expected to have as an NLP researcher/engineer.

### Exercise 00: Check how the data look like.
- Print out the first 30 lines of the corpus.

In [2]:
# write your command here:
# print the first 30 lines of the corpus by linux command
!head -n 30 "UD_English-GUM/en_gum-ud-train.conllu"

# newdoc id = GUM_academic_art
# global.Entity = GRP-etype-infstat-salience-centering-minspan-link-identity
# meta::author = Claire Bailey-Ross, Andrew Beresford, Daniel Smith, Claire Warwick
# meta::dateCollected = 2017-09-13
# meta::dateCreated = 2017-08-08
# meta::dateModified = 2017-09-13
# meta::genre = academic
# meta::salientEntities = 4 (5*), 5 (5*), 44 (5*), 45 (5*), 46 (5*), 47 (5*), 27 (4*), 147 (4*), 2 (3*), 43 (3), 20 (2*), 23 (2), 63 (2), 72 (2), 73 (2), 3 (1), 19 (1), 24 (1), 26 (1), 48 (1), 49 (1), 50 (1), 62 (1), 68 (1), 69 (1), 74 (1), 76 (1), 77 (1), 78 (1), 79 (1), 158 (1)
# meta::sourceURL = https://dh2017.adho.org/abstracts/333/333.pdf
# meta::speakerCount = 0
# meta::summary1 = (human) This paper presents an eye tracking study to explore how viewers experience art, focusing on a 17th Century collection of Spanish paintings by Zurbarán.
# meta::summary2 = (claude-3-5-sonnet-20241022) This pilot study uses eye-tracking techniques to examine how viewers visually pro

### Exercise 01: Count the number of sentences.



In [3]:
# write your command here:
# hint: use the command filter and then count the number of lines
!grep -E '^\#\s*text\s*=' "UD_English-GUM/en_gum-ud-train.conllu" | wc -l

10224


### Exercise 02: Count the number of tokens using command

In [4]:
# write your command here:
# hint: pay attention to how the numbers in the first column look like
# hint: filter with a regular expression
!grep -E '^[0-9]+(\.[0-9]+)?\t' "UD_English-GUM/en_gum-ud-train.conllu" | wc -l

0


### Exercise 03: Get the tagset.
- Get the unique instances of the POS labels.

In [5]:
# write your command here:
# hint: extract 4th column (tab separated file)
!grep -E '^[0-9]+(\.[0-9]+)?\t' "UD_English-GUM/en_gum-ud-train.conllu" | cut -f4 | sort -u

### Exercise 04: Get the frequency of each tag.

In [6]:
# write your command here:
# hint: check out available options of uniq: man uniq
!grep -E '^[0-9]+(\.[0-9]+)?\t' "UD_English-GUM/en_gum-ud-train.conllu" | cut -f4 | sort | uniq -c | sort -nr

## Preprocessing UD data in Python
To load this dataset in Python, we are going to use a library called `conllu`.
We first need to install the library using the following command.

In [7]:
!pip install conllu

We then define a function that read the file and convert it to a list of sentences, where each sentence is a list of tuples.
e.g.
`[[('MBZUAI', 'PROPN'), ('is', 'AUX'), ('a', 'DET'), ...], [...], ...]`.

In [8]:
from conllu import parse_incr

# define a function that reads the conllu format file
def read_conll(fpath) -> list:
    instances = []
    with open(fpath, mode='r') as f:
        for sentence in parse_incr(f):
            instances.append(
                [(token['form'], token['upos']) # we use universal pos for this exercise, alternatively, you can use language specific pos (xpos)
                  for token in sentence if type(token['id']) is int] # we need this to exclude MWEs
            )
    return instances

Now, we can load the file and get the statistics for sanity check.

In [9]:
# load the file using the above function
train = read_conll('UD_English-GUM/en_gum-ud-train.conllu')

In [10]:
# check the output
train[0]

[('Aesthetic', 'ADJ'),
 ('Appreciation', 'NOUN'),
 ('and', 'CCONJ'),
 ('Spanish', 'ADJ'),
 ('Art', 'NOUN'),
 (':', 'PUNCT')]

In [11]:
# number of sentences
len(train)

10224

In [12]:
import itertools # we need this to flatten the list

# number of tokens
len(list(itertools.chain.from_iterable(train)))

177410

# **Hidden Markov Model (HMM)**

## Background
### Bayse Rule

$$
P(A|B) = \frac{P(B|~A)P(A)}{P(B)}
$$

### Generative Sequence Model

Given a sentence $X$, predict its part of speech
sequence $Y$.
Find the most probable tag sequence give the sentence

$$
\underset{Y}{\operatorname{argmax}} P(Y|X)
$$

We first decompose probability using the Bayes rule.

$$
\begin{align}
\underset{Y}{\operatorname{argmax}} P(Y|X) &= \underset{Y}{\operatorname{argmax}} \frac{P(X|Y)P(Y)}{P(X)} \\
&= \underset{Y}{\operatorname{argmax}} P(X|Y)P(Y)
\end{align}
$$

Here, $P(X|Y)$ is the model of word/POS interactions ("this word is probably that tag"), and $P(Y)$ is the model of POS/POS interactions ("this tag comes after that tag").

## Training HMM

### Exercise 05: Count the number of occurances in the courpus

Create three dictionaries that store the number of occurances in the courpus.
- `transition_count`: dictionary that stores counts of co-occurances of the current tag and the previous (context) tag.
```python
# expected output
{'<s> ADJ': 246,
  'ADJ NOUN': 4872,
  'NOUN CCONJ': 1388,
  'CCONJ ADJ': 429,
  'NOUN PUNCT': 6791,
  ...}
```
- `emission_count`: dictionary that stores counts of co-occurance of the current tag and its word.
```python
# expected output
{'ADJ Aesthetic': 1,
'NOUN Appreciation': 1,
'CCONJ and': 2947,
'ADJ Spanish': 14,
'NOUN Art': 1,
...}
```
- `context_count`: dictionary that stores counts of context tags.
```python
# expected output
{'<s>': 6917,
'ADJ': 8341,
'NOUN': 21672,
'CCONJ': 4066,
'PUNCT': 16973,
...}
```

Fill out `'???'` and complete the following code snipet.

In [13]:
from collections import defaultdict

# initialize dictionaries
transition_count = defaultdict(int)
emission_count = defaultdict(int)
context_count = defaultdict(int)

for sentence in train:
    # we start from the initial symbol
    previous = '<s>'
    context_count[previous] += 1
    for token in sentence:
        word, tag = token[0], token[1]
        transition_count[f'{previous} {tag}'] += 1
        context_count[tag] += 1
        emission_count[f'{tag} {word}'] += 1
        previous = tag
    transition_count[f'{previous} </s>'] += 1

In [14]:
transition_count

defaultdict(int,
            {'<s> ADJ': 374,
             'ADJ NOUN': 6599,
             'NOUN CCONJ': 1854,
             'CCONJ ADJ': 553,
             'NOUN PUNCT': 9455,
             'PUNCT </s>': 9560,
             '<s> NOUN': 543,
             'NOUN ADP': 6794,
             'ADP NOUN': 3047,
             'PUNCT NOUN': 1164,
             'NOUN </s>': 292,
             '<s> PROPN': 801,
             'PROPN PROPN': 2551,
             'PROPN PUNCT': 3057,
             'PUNCT PROPN': 1340,
             'PROPN ADP': 942,
             'ADP PROPN': 1912,
             'PUNCT VERB': 1195,
             'VERB PROPN': 444,
             'PROPN </s>': 205,
             '<s> ADV': 894,
             'ADV AUX': 325,
             'AUX NOUN': 154,
             'NOUN VERB': 1821,
             'VERB ADP': 3752,
             'ADP CCONJ': 83,
             'CCONJ VERB': 875,
             'VERB NOUN': 1822,
             '<s> DET': 1022,
             'DET NOUN': 8172,
             'ADP ADJ': 1427,
        

In [15]:
emission_count

defaultdict(int,
            {'ADJ Aesthetic': 1,
             'NOUN Appreciation': 1,
             'CCONJ and': 4022,
             'ADJ Spanish': 14,
             'NOUN Art': 1,
             'PUNCT :': 336,
             'NOUN Insights': 1,
             'ADP from': 598,
             'NOUN Eye': 1,
             'PUNCT -': 928,
             'NOUN Tracking': 1,
             'PROPN Claire': 2,
             'PROPN Bailey': 2,
             'PROPN Ross': 4,
             'PROPN claire.bailey-ross@port.ac.uk': 1,
             'PROPN University': 61,
             'ADP of': 4126,
             'PROPN Portsmouth': 1,
             'PUNCT ,': 9497,
             'VERB United': 84,
             'PROPN Kingdom': 8,
             'PROPN Andrew': 6,
             'PROPN Beresford': 2,
             'PROPN a.m.beresford@durham.ac.uk': 1,
             'PROPN Durham': 5,
             'PROPN Daniel': 24,
             'PROPN Smith': 4,
             'PROPN daniel.smith2@durham.ac.uk': 1,
             'PROPN Warwic

In [16]:
context_count

defaultdict(int,
            {'<s>': 10224,
             'ADJ': 11489,
             'NOUN': 29289,
             'CCONJ': 5854,
             'PUNCT': 24563,
             'ADP': 16655,
             'PROPN': 10144,
             'VERB': 18642,
             'ADV': 8556,
             'AUX': 9682,
             'DET': 14307,
             'PRON': 15197,
             'SCONJ': 2878,
             'X': 331,
             'SYM': 281,
             'PART': 4314,
             'NUM': 3368,
             'INTJ': 1860})

### Exercise 06: Compute transition probabilities and emission probabilities



Create two dictionaries that store transition probabitlies and emission probabitlies.

- `transition_prob`: dictionary that stores the probabilities of transition from the previous tag to the current tag.
```python
# expected output
{'<s> ADJ': 0.035564551105970794,
'ADJ NOUN': 0.5841026255844622,
'NOUN CCONJ': 0.06404577334809892,
'CCONJ ADJ': 0.10550909985243483,
'NOUN PUNCT': 0.3133536360280546,
...}
```
- `emission_prob`: dictionary that stores the probabilities of a word being generated on its own tag.
```python
# expected output
{'ADJ Aesthetic': 0.00011988970147464332,
'NOUN Appreciation': 4.614248800295312e-05,
'CCONJ and': 0.7247909493359567,
'ADJ Spanish': 0.0016784558206450065,
'NOUN Art': 4.614248800295312e-05,
...}
```

Fill out `'???'` and complete the following code snipet.

In [17]:
# initialize dictionaries
transition_prob = defaultdict(float)
emission_prob = defaultdict(float)

for transition, count in transition_count.items():
    previous = transition.split(' ', 1)[0]
    transition_prob[transition] = count / context_count[previous]

for emission, count in emission_count.items():
    tag = emission.split(' ', 1)[0]
    emission_prob[emission] = count / context_count[tag]

In [18]:
transition_prob

defaultdict(float,
            {'<s> ADJ': 0.03658059467918623,
             'ADJ NOUN': 0.5743754895987466,
             'NOUN CCONJ': 0.06330021509781829,
             'CCONJ ADJ': 0.09446532285616673,
             'NOUN PUNCT': 0.3228174399945372,
             'PUNCT </s>': 0.3892032732158124,
             '<s> NOUN': 0.053110328638497656,
             'NOUN ADP': 0.23196421864863942,
             'ADP NOUN': 0.18294806364455118,
             'PUNCT NOUN': 0.0473883483287872,
             'NOUN </s>': 0.00996961316535218,
             '<s> PROPN': 0.07834507042253522,
             'PROPN PROPN': 0.2514787066246057,
             'PROPN PUNCT': 0.3013604100946372,
             'PUNCT PROPN': 0.05455359687334609,
             'PROPN ADP': 0.0928627760252366,
             'ADP PROPN': 0.11480036025217652,
             'PUNCT VERB': 0.04865040915197655,
             'VERB PROPN': 0.023817186997103314,
             'PROPN </s>': 0.020208990536277602,
             '<s> ADV': 0.087441314553

In [19]:
emission_prob

defaultdict(float,
            {'ADJ Aesthetic': 8.703977717817043e-05,
             'NOUN Appreciation': 3.414251084024719e-05,
             'CCONJ and': 0.6870515886573283,
             'ADJ Spanish': 0.001218556880494386,
             'NOUN Art': 3.414251084024719e-05,
             'PUNCT :': 0.013679110857794243,
             'NOUN Insights': 3.414251084024719e-05,
             'ADP from': 0.03590513359351546,
             'NOUN Eye': 3.414251084024719e-05,
             'PUNCT -': 0.03778040141676505,
             'NOUN Tracking': 3.414251084024719e-05,
             'PROPN Claire': 0.0001971608832807571,
             'PROPN Bailey': 0.0001971608832807571,
             'PROPN Ross': 0.0003943217665615142,
             'PROPN claire.bailey-ross@port.ac.uk': 9.858044164037855e-05,
             'PROPN University': 0.006013406940063091,
             'ADP of': 0.24773341338937255,
             'PROPN Portsmouth': 9.858044164037855e-05,
             'PUNCT ,': 0.386638439929976,
         

## Finding POS Tags: Viterbi Algorithm

### Forward step
- Find the path to each node with the lowest negative log probability (highest probability)
- First, calculate transition from `<s>` and emission of the
first word for every POS.
- For middle words, calculate the minimum score for all
possible previous POS tags.
- Finish up the sentence with the sentence final symbol

#### Exercise 07: Annotate code to understand
- Add comments (`# like this`) to describe what is happening in each line

In [20]:
from math import log2  # import log base-2 for computing negative log-probability scores

possible_tags = context_count.keys()  # set of all tags seen in training, including '<s>' and maybe '</s>'
# hyperparameters for linear interpolation for emission probability of unknown words
lam = 0.95  # interpolation weight for known emission probability
vocab_size = 1e6  # pseudo-vocabulary size for unknown word smoothing

def forward(tokens):  # Viterbi-style forward pass to compute best path scores and backpointers
    # initialize dictionaries to store best_score and the edge
    best_score = {}  # maps "time_index tag" -> best (lowest) score found so far
    best_edge = {}   # maps "time_index tag" -> previous "time_index tag" that yields the best score

    # we start from timestep 0 with the label <s>
    # our goal is find the best path with the best score at the end
    best_score['0 <s>'] = 0  # score 0 means log-probability 1 at the start
    best_edge['0 <s>'] = None  # start has no previous edge

    for i in range(len(tokens)):  # iterate over token positions
        for prev in possible_tags:  # consider all possible previous tags
            for current in possible_tags:  # consider all possible current tags
                # proceed only if a path to (i, prev) exists and the transition prev->current is possible
                if f'{i} {prev}' in best_score and f'{prev} {current}' in transition_prob:
                    prob_t = transition_prob[f'{prev} {current}']  # transition probability P(current | prev)
                    # emission with interpolation for unknowns: P_e = λ*P(word|current) + (1-λ)/|V|
                    prob_e = lam * emission_prob[f'{current} {tokens[i]}'] + (1 - lam) / vocab_size
                    # accumulate cost as sum of negative log-probabilities
                    score = best_score[f'{i} {prev}'] + -log2(prob_t) + -log2(prob_e)

                    # update if this path gives a better (lower) score for state (i+1, current)
                    if f'{i+1} {current}' not in best_score or score < best_score[f'{i+1} {current}']:
                        # update the best score
                        best_score[f'{i+1} {current}'] = score  # record improved score
                        # add an edge
                        best_edge[f'{i+1} {current}'] = f'{i} {prev}'  # backpointer to the best predecessor

    # do the same thing for the end of sentence
    for tag in possible_tags:  # finalize by transitioning from last token position to </s>
        # only if we reached (len(tokens), tag) and the transition tag-></s> exists
        if f'{len(tokens)} {tag}' in best_score and f'{tag} </s>' in transition_prob:
            # add cost of transitioning to </s>
            score = best_score[f'{len(tokens)} {tag}'] - log2(transition_prob[f'{tag} </s>'])
            # update best end score if improved
            if f'{len(tokens)+1} </s>' not in best_score or best_score[f'{len(tokens)+1} </s>'] > score:
                best_score[f'{len(tokens)+1} </s>'] = score  # best complete-path score
                best_edge[f'{len(tokens)+1} </s>'] = f'{len(tokens)} {tag}'  # backpointer to last tag

    return best_edge  # return backpointers for path reconstruction

###  Backward Step of Viterbi Algorithm
- Reproduce the path.

In [21]:
def backward(tokens, best_edge):
    tags = list()
    next_edge = best_edge[f'{len(tokens)+1} </s>']

    # we loop until we hit the start
    while next_edge != "0 <s>":
        # add the substring for this edge to the words
        position, tag = next_edge.split()
        tags.append(tag)
        next_edge = best_edge[next_edge]

    return tags[::-1]

We put the forward and backward functions together.

In [22]:
def hmm_tagger(tokens):
    best_edge = forward(tokens)
    tags = backward(tokens, best_edge)
    return tags

In [23]:
example_tokens = 'MBZUAI is a graduate research university dedicated to advancing AI as a global force for good .'.split()
hmm_tagger(example_tokens)

['PRON',
 'AUX',
 'DET',
 'ADJ',
 'NOUN',
 'NOUN',
 'VERB',
 'PART',
 'VERB',
 'NOUN',
 'ADP',
 'DET',
 'ADJ',
 'NOUN',
 'ADP',
 'ADJ',
 'PUNCT']

### Evaluation

Now that we built our HMM tagger, we can evluate its performance on the dev set.
We first read the dev file.

In [24]:
dev = read_conll('UD_English-GUM/en_gum-ud-dev.conllu')

#### Exercise 08: Prepare input for HMM tagger and gold labels for evaluation
Fill out `'???'` and complete the following code snipet.

In [25]:
dev[0]

[('Introduction', 'NOUN')]

In [26]:
# list of list of words, dev[0] is a list containing tuples of (word, label)
dev_sents = [[x for x,y in sent] for sent in dev]
dev_labels = [[y for x,y in sent] for sent in dev]

Now we can run our HMM tagger on the dev set.

In [27]:
dev_preds = [hmm_tagger(sent) for sent in dev_sents]

#### Exercise 09: Write a function that calculates accuracy

In [28]:
def accuracy(pred, gold) -> float:
    acc = 0
    total = 0
    # do something here
    for pred, gold in zip(pred, gold):
        for p, g in zip(pred, gold):
            if p == g:
                acc += 1
            total += 1
    return acc / total

In [29]:
accuracy(dev_preds, dev_labels)

0.9078558981471603

# Other NLP tools for POS tagging

## Stanza (Multi-lingual)

![image](https://github.com/stanfordnlp/stanza/raw/dev/images/stanza-logo.png)
  > Stanza is a Python natural language analysis package. It contains tools, which can be used in a pipeline, to convert a string containing human language text into lists of sentences and words, to generate base forms of those words, their parts of speech and morphological features, to give a syntactic structure dependency parse, and to recognize named entities.

- [Stanza: A Python Natural Language Processing Toolkit for Many Human Languages](https://aclanthology.org/2020.acl-demos.14/) (Qi et al., ACL 2020)
- [GitHub](https://github.com/stanfordnlp/stanza)

In [30]:
# install stanza
!pip install stanza

  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached triton-3.6.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (1.7 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached colorama-0.4.6-py2.py3-none-any.whl.metadata (17 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 27.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.6/530.6 MB 25.8 MB/s  0:00:18m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 29.5 MB/s  0:00:11m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 87.5 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 87.4 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 64.0 MB/s  0:00:00 eta 0:00:01m
Using cached triton-3.6.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (188.2 MB)
   ━━━━━━━━━

Let's try the POS tagger implemented in Stanza and evaluate the POS tagging accuracy and compare with our HMM tagger.
**However, this is not strictly a fair comparison. Why?**


In [31]:
import stanza

# instantiate the tagger
stanza_tagger = stanza.Pipeline(lang='en',
                                processors='tokenize,mwt,pos',
                                tokenize_pretokenized=True)

/home/khang.nhat/anaconda3/envs/nlplab/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-28 06:44:25 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2026-03-28 06:44:25 INFO: Downloaded file to /home/khang.nhat/.cache/stanza/1.11.0/resources/resources.json
2026-03-28 06:44:30 INFO: Loading these models for language: en (English):
| Processor | Package         |
-------------------------------
| tokenize  | combined        |
| mwt       | combined        |
| pos       | combined_charlm |

/home/khang.nhat/anaconda3/envs/nlplab/lib/python3.11/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too

In [32]:
# tag the sentences and extract labels we are interested in
stanza_out = stanza_tagger(dev_sents)
dev_preds = [[word.upos for word in sent.words] for sent in stanza_out.sentences]

In [33]:
# calculate the accuracy
accuracy(dev_preds, dev_labels)

0.9801557665635335

## CAMeL Tools (Language-specific: Arabic)
![image](https://github.com/CAMeL-Lab/camel_tools/raw/master/camel_tools_logo.png)
> CAMeL Tools is a collection of open-source tools for Arabic natural language processing in Python.
CAMeL Tools currently provides utilities for pre-processing, morphological modeling, Dialect Identification, Named Entity Recognition and Sentiment Analysis.

- [CAMeL Tools: An Open Source Python Toolkit for Arabic Natural Language Processing](https://aclanthology.org/2020.lrec-1.868/) (Obeid et al., LREC 2020)
- Guided tour (Jupyter Notebook) of CAMeL Tools is avaiable [here](https://colab.research.google.com/drive/1Y3qCbD6Gw1KEw-lixQx1rI6WlyWnrnDS?usp=sharing).

First, we install CAMeL Tools and download a tagging model.

In [34]:
# !CMAKE_OSX_ARCHITECTURES=arm64 pip install camel-tools
!pip install camel-tools

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached pyyaml-6.0.3-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
  Using cached safetensors-0.7.0-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 5.9 MB/s  0:00:00 eta 0:00:01m
Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.3 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 11.9 MB/s  0:00:00 eta 0:00:01
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 44.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6

In [35]:
!camel_data -i disambig-bert-unfactored-msa

The following packages will be installed: 'morphology-db-msa-r13', 'disambig-bert-unfactored-msa', 'disambig-ranking-cache-calima-msa-r13'
Extracting package 'morphology-db-msa-r13': 100%|█| 40.5M/40.5M [00:00<00:00, 1.
Extracting package 'disambig-bert-unfactored-msa': 100%|█| 445M/445M [00:00<00:0
Extracting package 'disambig-ranking-cache-calima-msa-r13': 100%|█| 556M/556M [0


Let's try with an example.
We first tokenize the input and run the tagger.

In [36]:
from camel_tools.disambig.bert import BERTUnfactoredDisambiguator

# initialize the model
unfactored = BERTUnfactoredDisambiguator.pretrained()

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]


Some weights of the model checkpoint at /home/khang.nhat/.camel_tools/data/disambig_bert_unfactored/msa were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [37]:
from camel_tools.tokenizers.word import simple_word_tokenize

# tokenize
tokens = simple_word_tokenize('جامعة محمد بن زايد للذكاء الاصطناعي هي جامعة بحثية للدراسات العليا مكرسة لتعزيز الذكاء الاصطناعي')
tokens

['جامعة',
 'محمد',
 'بن',
 'زايد',
 'للذكاء',
 'الاصطناعي',
 'هي',
 'جامعة',
 'بحثية',
 'للدراسات',
 'العليا',
 'مكرسة',
 'لتعزيز',
 'الذكاء',
 'الاصطناعي']

In [38]:
# run the tagger
disambiguated = unfactored.disambiguate(tokens)

# extract the top analysis for each token
# an analysis contains different types of information
# such as pos, lex, diacritics, proclitics, etc
analyses = [d.analyses[0].analysis for d in disambiguated]
analyses

[{'diac': 'جامِعَةُ',
  'lex': 'جامِعَة',
  'bw': 'جامِع/NOUN+َة/NSUFF_FEM_SG+ُ/CASE_DEF_NOM',
  'gloss': 'university;league+[fem.sg.]',
  'pos': 'noun',
  'prc3': '0',
  'prc2': '0',
  'prc1': '0',
  'prc0': '0',
  'per': 'na',
  'asp': 'na',
  'vox': 'na',
  'mod': 'na',
  'stt': 'c',
  'cas': 'n',
  'enc0': '0',
  'rat': 'i',
  'source': 'lex',
  'form_gen': 'f',
  'form_num': 's',
  'd3seg': 'جامِعَةُ',
  'caphi': 'j_aa_m_i_3_a_t_u',
  'd1tok': 'جامِعَةُ',
  'd2tok': 'جامِعَةُ',
  'pos_logprob': -0.4344233,
  'd3tok': 'جامِعَةُ',
  'd2seg': 'جامِعَةُ',
  'pos_lex_logprob': -3.240683,
  'num': 's',
  'ud': 'NOUN',
  'gen': 'f',
  'catib6': 'NOM',
  'root': 'ج.م.ع',
  'bwtok': 'جامِع_+َة_+ُ',
  'pattern': '1ا2ِ3َةُ',
  'lex_logprob': -3.240683,
  'atbtok': 'جامِعَةُ',
  'atbseg': 'جامِعَةُ',
  'd1seg': 'جامِعَةُ',
  'stem': 'جامِع',
  'stemgloss': 'university;league',
  'stemcat': 'NapAt'},
 {'diac': 'مُحَمَّد',
  'lex': 'مُحَمَّد',
  'bw': 'مُحَمَّد/NOUN_PROP',
  'gloss': 'Muhammad;

In [39]:
# extract the labels we're interested in
pos = [d.analyses[0].analysis['pos'] for d in disambiguated]
pos

['noun',
 'noun_prop',
 'noun_prop',
 'noun_prop',
 'noun',
 'adj',
 'pron',
 'noun',
 'adj',
 'noun',
 'adj',
 'adj',
 'noun',
 'noun',
 'adj']

# Acknowledgement
Part of this lab material is based on Gram Neubig's NLP tutorial: http://www.phontron.com/slides/nlp-programming-en-04-hmm.pdf
